<table align="left">
  <td>
    <a href="https://colab.research.google.com/github/phonchi/CryoParticleSegment/blob/main/notebook/04_particle_extraction_clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>
  </td>
</table>

### CryoParticleSegment

In [1]:
do = False # @param{type:"boolean"}
if do:
    %pip install torchinfo -qq
    %pip install -U git+https://github.com/qubvel/segmentation_models.pytorch -qq
    %pip install starfile -qq
    %pip install https://github.com/soft-matter/trackpy/archive/master.zip -qq

In [2]:
if do:
    !git clone https://github.com/phonchi/CryoParticleSegment.git

    !wget -O /content/CryoParticleSegment/Modeling/convcrf.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/convcrf.py
    !wget -O /content/CryoParticleSegment/Modeling/dataset.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/dataset.py
    !wget -O /content/CryoParticleSegment/Modeling/model.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/model.py
    !wget -O /content/CryoParticleSegment/Modeling/trainer.py https://raw.githubusercontent.com/SRT-0/CPS_modeling_adjusted_for_denoise_CRF/main/adjusted_modeling/trainer.py

In [3]:
import sys
import os

# Adjust the path relative to your current working directory
module_path = os.path.abspath('CryoParticleSegment/Modeling')

# Add to sys.path if it's not already included
if module_path not in sys.path:
    sys.path.append(module_path)

In [4]:
if do:
    #%git clone https://github.com/netw0rkf10w/CRF.git
    %cd CryoParticleSegment/Modeling/CRF_main
    !python setup.py clean --all
    !rm -rf build/
    !python setup.py build_ext --inplace --force
    !python setup.py install

> #### ⚠ Notice
>
> You need to restart the kernel after the following step.

In [5]:
if do:
    %pip install pycuda==2024.1
    %pip install "numpy<2.0"
    %pip install mrcfile -qq

## ⭐ Setup
You must run all codes under this category.

### ✅ Directory Settings

In [6]:
# @title  { display-mode: "form" }

INPUT_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
IMAGE_DIR = INPUT_IMAGE_DIR
# @markdown ---

use_denoised_as_pariwise = False # @param {type : "boolean"}
dnzd_pw = use_denoised_as_pariwise
DENOISED_IMAGE_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/dataset/10017/processed_micrographs_np_split" # @param {type:"string"}
# @markdown ---

LABEL_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset/10017/micrographs_ground_np" # @param {type:"string"}
DATASET_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output/dataset" # @param {type:"string"}
RESULT_DIR = "/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_diff_post_processing_but_circularity_MAD/10017/unet_eb5_dice_CRF" # @param {type:"string"}

In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# @title  { display-mode: "form" }
# @markdown Detect whether using folder in Google Drive as **`RESULT DIR`**📁.
import os
if "content" in IMAGE_DIR.split("/")[:3] or "content" in LABEL_DIR.split("/")[:3]:
  try:
    !rm -r /content/sample_data
    if not os.path.exists("/content/image_dir"):
        if "content" in IMAGE_DIR.split("/")[:3]:
            !cp -r {IMAGE_DIR} /content/image_dir
            IMAGE_DIR = "/content/image_dir"
        if "content" in LABEL_DIR.split("/")[:3]:
            !cp -r {LABEL_DIR} /content/label_dir
            LABEL_DIR = "/content/label_dir"
  except:
    pass

In [9]:
IMAGE_DIR = "/content/image_dir"

In [10]:
%cd /content/

/content


### ✅ Packages Handling

In [11]:
# @title  { display-mode: "form" }
# @markdown Useful packages.

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau, OneCycleLR

In [12]:
# @title  { display-mode: "form" }
# @markdown User-defined packages.

from dataset import MicrographDataset, MicrographDatasetEvery
from dataset import reconstruct_patched, collate_fn
from model import create_model
from trainer import CryoEMEvaluator
from trainer import CryoEMTrainerWithScheduler, tqdm_plugin_for_Trainer

## ⭐ Main

### ✅ Setting

In [13]:
# @markdown Parameters.

NUM_CLASSES = 2
EPOCHS = 50
BATCH = 8
CROP_SIZE = (512, 512)
LR = 1e-3

RLR_PATIENCE = 3
ES_PATIENCE = 15
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [14]:
# @markdown Set seed.

random_state = 42
torch.manual_seed(random_state)
torch.cuda.manual_seed_all(random_state)

## ⭐ Convcrf wtih FCN finetuned on cryoem

### ✅ Model

## The model

In [15]:
# @title  { display-mode: "form" }

architecture = "Unet++" # @param {type:"string"}
encoder = "timm-efficientnet-b5" # @param {type:"string"}
pretrained = True # @param {type:"boolean"}
solver = "fw" # @param {type:"string"}
use_unary_only = False # @param {type:"boolean"}


In [16]:
import segmentation_models_pytorch as smp

if pretrained:
  weights = "imagenet"
else:
  weights = None

if architecture == "Unet++":
    backbone = smp.UnetPlusPlus(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` or `advprop` for pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )

elif architecture == "Deeplab":
    backbone = smp.DeepLabV3(
        encoder_name=encoder,        # choose encoder, densenet201, resnet50, e.g. mobilenet_v2 or efficientnet-b5
        encoder_weights=weights,     # use `imagenet` pre-trained weights for encoder initialization
        in_channels=1,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
        classes=2,                      # model output channels (number of classes in your dataset)
    )
else:
    print("Architecture not supported")
    raise NotImplementedError

model = create_model(backbone, addout=True) #crf_args

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:86: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/122M [00:00<?, ?B/s]

In [17]:
import CRF
import torch.nn as nn
from model import setup_crf, create_fwcrf_model

# Example usage
solver = 'fw'  # Assuming the solver type is defined

crf = setup_crf(solver, NUM_CLASSES)
model_post = create_fwcrf_model(model.backbone, crf, use_unary_only=True)

CRF solver: fw
x0_weight: 0.0
FrankWolfeParams: 
	 scheme:	 fixed 
	 stepsize:	 1.0 (for the 'fixed' scheme) 
	 regularizer:	 l2
	 lambda_:	 1.0
	 lambda_learnable:	 False
	 x0_weight:	 0.5
	 x0_weight_learnable:	 False
Non-trainable lambda for Frank-Wolfe: 1.0
Non-trainable x0_weight for Frank-Wolfe: 0.5
Potts: remove random weights.
Add 1.0 to spatial_weight diagonal
Add 1.0 to bilateral_weight diagonal
Add -1.0 to compatibility diagonal


## ⭐ Evaluate

In [18]:
import gc
gc.collect()
torch.cuda.empty_cache()

from torchvision.utils import save_image
from dataset import reconstruct_patched

def simple_micrograph_preprocessing(micrograph):
  micrograph_copy = micrograph.copy()
  micrograph_copy = (micrograph_copy-micrograph.mean()+2.5*micrograph.std())/5/micrograph.std()
  micrograph_copy[micrograph_copy<0]=0
  micrograph_copy[micrograph_copy>1]=1
  return micrograph_copy

In [19]:
from torch.utils.data import ConcatDataset

train_dir = os.path.join(IMAGE_DIR, 'train')
train_filenames = np.loadtxt(f"{IMAGE_DIR}/train_filenames.txt", dtype=str)
if dnzd_pw == False:
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, filenames=train_filenames, crop_size=CROP_SIZE)
else:
    dnzd_train_dir = os.path.join(DENOISED_IMAGE_DIR, 'train')
    train_dataset = MicrographDatasetEvery(image_dir=train_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_train_dir, filenames=train_filenames, crop_size=CROP_SIZE)
train_loader = DataLoader(train_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


val_dir = os.path.join(IMAGE_DIR, 'val')
val_filenames = np.loadtxt(f"{IMAGE_DIR}/val_filenames.txt", dtype=str)
if dnzd_pw == False:
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, filenames=val_filenames, crop_size=CROP_SIZE)
else:
    dnzd_val_dir = os.path.join(DENOISED_IMAGE_DIR, 'val')
    val_dataset = MicrographDatasetEvery(image_dir=val_dir, label_dir=LABEL_DIR, denoised_dir = dnzd_val_dir, filenames=val_filenames, crop_size=CROP_SIZE)
val_loader = DataLoader(val_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)

test_dir = os.path.join(IMAGE_DIR, 'test')
test_filenames = np.loadtxt(f"{IMAGE_DIR}/test_filenames.txt", dtype=str)
if dnzd_pw == False:
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, filenames=test_filenames, crop_size=CROP_SIZE)
else:
    dnzd_test_dir = os.path.join(DENOISED_IMAGE_DIR, 'test')
    test_dataset = MicrographDatasetEvery(image_dir=test_dir, label_dir=None, denoised_dir = dnzd_test_dir, filenames=test_filenames, crop_size=CROP_SIZE)

test_loader = DataLoader(test_dataset, batch_size=None, shuffle=False, pin_memory=True, collate_fn=collate_fn)


full_filenames = np.concatenate((train_filenames, val_filenames, test_filenames))
full_dataset = ConcatDataset([train_dataset, val_dataset, test_dataset])
full_loader = DataLoader(full_dataset, batch_size=32, shuffle=True, pin_memory=True, collate_fn=collate_fn)

In [20]:
shape = None
for i1, i2, i3, i4, i5 in val_loader: #test loader and reconstruct
    shape = i5.shape
    break

In [21]:
from PIL import Image

def normalize(im):
    max_mrc=np.max(im)
    min_mrc=np.min(im)
    img_original=(255*((im-min_mrc)/(max_mrc-min_mrc))).astype(np.uint8)
    return(img_original)

def preprocess_and_crop(micrograph, crop_size=3840):
    processed_micrograph = simple_micrograph_preprocessing(micrograph)
    if crop_size:
        mic_width, mic_height = processed_micrograph.shape[1], processed_micrograph.shape[0]
        start_x, start_y = (mic_width - crop_size) // 2, (mic_height - crop_size) // 2
        end_x, end_y = start_x + crop_size, start_y + crop_size
        return processed_micrograph[start_y:end_y, start_x:end_x]
    else:
        return processed_micrograph

In [22]:
from tqdm import tqdm

fnames = []
n = len(full_dataset)
processed_micrographs = np.empty((n, shape[1], shape[2]), dtype=np.float32)

# Use tqdm to wrap your loop for a progress bar
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing images"):
    name = full_filenames[idx][:-4]
    fnames.append(name)
    # Determine the directory and load the micrograph
    if os.path.exists(f"{IMAGE_DIR}/test/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/test/{name}.npy"
    elif os.path.exists(f"{IMAGE_DIR}/train/{name}.npy"):
        micrograph_path = f"{IMAGE_DIR}/train/{name}.npy"
    else:
        micrograph_path = f"{IMAGE_DIR}/val/{name}.npy"
    micrograph = np.load(micrograph_path)
    processed_micrograph = preprocess_and_crop(micrograph)
    # Place the processed micrograph directly into the preallocated array
    processed_micrographs[idx] = processed_micrograph

Processing images: 100%|██████████| 84/84 [00:27<00:00,  3.02it/s]


In [23]:
processed_micrographs.shape

(84, 3840, 3840)

In [24]:
np.save(f"{RESULT_DIR}/processed_micrographs.npy", processed_micrographs)

In [25]:
processed_micrographs = np.load(f"{RESULT_DIR}/processed_micrographs.npy")

In [26]:
del model
torch.cuda.empty_cache()

In [27]:
model = model_post

In [28]:
RESULT_DIR

'/content/drive/MyDrive/cryo_project/Test_on_10017/Test_0_9-8_9-30_EM_10017/raw_user_output_rst/results_diff_post_processing_but_circularity_MAD/10017/unet_eb5_dice_CRF'

In [29]:
import torch.nn.functional as F

label_images = np.empty((0, shape[1], shape[2]), dtype=np.uint8)

checkpoint_paths = [path for path in os.listdir(RESULT_DIR) if '.pt' in path]
checkpoint_path = checkpoint_paths[-1]
state_dict_path = f"{RESULT_DIR}/{checkpoint_path}"
state_dict = torch.load(state_dict_path, map_location=torch.device(device))
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

mini_batch_size = 18  # Number of patches to process at once
n = len(full_dataset)
label_images = np.empty((n, shape[1], shape[2]), dtype=np.uint8)  # Preallocated array

# Iterate through the dataset with tqdm for progress tracking
for idx, (test_image, _, _, grid, _) in tqdm(enumerate(full_dataset), total=n, desc="Processing dataset"):
    # Process in batches
    with torch.no_grad():
        inputs = test_image.to(device)
        num_batches = (inputs.size(0) + mini_batch_size - 1) // mini_batch_size
        patched_outputs = []

        for batch_idx in range(num_batches):
            start_idx = batch_idx * mini_batch_size
            end_idx = min(start_idx + mini_batch_size, inputs.size(0))
            patch_input = inputs[start_idx:end_idx].to(device)
            output = model(patch_input)['out']
            patched_outputs.append(output.cpu())  # Minimize device memory usage

            # Cleanup as soon as not needed
            del patch_input, output
            torch.cuda.empty_cache()

        outputs = torch.cat(patched_outputs)  # Combine outputs
        probabilities = F.softmax(outputs, dim=1)
        class1_probabilities = probabilities[:, 1, :, :]  # Assuming class 1 is the target
        pred_image = reconstruct_patched(class1_probabilities.unsqueeze(1), grid).float()

        output_image = normalize(pred_image.squeeze().numpy())

        # Cleanup large temporary variables
        del patched_outputs, outputs, probabilities, class1_probabilities, pred_image
        torch.cuda.empty_cache()

    # Store the output image directly in the preallocated array
    label_images[idx] = output_image

    if idx % 30 == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(output_image, cmap='inferno', alpha=0.4)
        plt.show()
    del output_image
    torch.cuda.empty_cache()

Output hidden; open in https://colab.research.google.com to view.

In [30]:
np.save(f"{RESULT_DIR}/label_images.npy", label_images)

In [31]:
!cp {RESULT_DIR}/label_images.npy .

In [32]:
label_images = np.load(f"{RESULT_DIR}/label_images.npy")

In [33]:
label_images.shape

(84, 3840, 3840)

In [34]:
# @title  { vertical-output: true, display-mode: "form" }
EMPIAR_ID = 10017 # @param {type:"integer"}
RADIUS = 64 # @param {type:"integer"}
# For 10017
BORDER = 128 # @param {type:"integer"}

In [35]:
!cp {RESULT_DIR}/best_config.txt .

In [36]:
with open("best_config.txt", "r") as f:
    for line in f:
        key, value = line.strip().split(": ", 1)
        if key == "cv_config":
            cv_config = eval(value)
        elif key == "tp_config":
            tp_config = eval(value)
        elif key == "nms_config":
            nms_config = eval(value)

---

In [40]:
import json

# Load the best parameters found
watershed_json = "best_watershed_params.json" # @param{type:"string"}
!cp {RESULT_DIR}/{watershed_json} .
with open(watershed_json, 'r') as f:
    best_params = json.load(f)

# changed) Extracting specific values from the parsed dictionary
best_thresh = best_params['peak_thresh']
best_dist = best_params['min_dist']

print(f"Loaded Config: peak_Thresh={best_thresh}, min_Dist={best_dist}")

ws_config = [best_thresh, best_dist]

Loaded Config: peak_Thresh=0.3, min_Dist=0.75


---

### Actual seg

In [41]:
radius = RADIUS

In [42]:
from skimage import filters, segmentation, morphology
import skimage as ski
from skimage import img_as_float
from center_finding import normalize, min_rect_circle, eliminate_near
import cv2

In [43]:
def get_mad_range(data, z_thresh=3.0):
    """Calculates the filtering window using Median Absolute Deviation."""
    arr = np.array(data)
    if len(arr) == 0: return 0.0, 1.0
    med = np.median(arr)
    mad = np.median(np.abs(arr - med))
    # 0.6745 is the scaling factor to make MAD consistent with standard deviation
    low = med - (z_thresh * mad / 0.6745)
    high = med + (z_thresh * mad / 0.6745)
    return max(0.0, low), min(1.0, high)

In [44]:
from skimage import img_as_float
from skimage.filters import threshold_local

cv_list = []
for img in label_images:
    output_image = img_as_float(img)
    block_size = int(round(radius * 1.6)) | 1  # Ensure it's an odd number
    local_thresh = filters.threshold_local(output_image, block_size, method='gaussian', offset=0)
    image_seg = output_image > local_thresh
    thresh1 = ski.util.img_as_ubyte(image_seg)

    contr_min = radius*cv_config[1]
    kernel_size = int(radius / cv_config[0])  # Example ratio
    kernel = np.ones((kernel_size, kernel_size), np.uint8)
    thresh1 = cv2.erode(thresh1, kernel, iterations=1)

    contours, hierarchy = cv2.findContours(thresh1,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    cont_array = [c for c in contours]

    # 1. Initial Area Filtering (changed)
    candidates = []
    circularities = []

    for cnt in contours:
        area = cv2.contourArea(cnt)
        if contr_min < area < 500000:
            perimeter = cv2.arcLength(cnt, True)
            if perimeter > 0:
                # Calculate circularity: (4 * pi * Area) / Perimeter^2
                circ = (4 * np.pi * area) / (perimeter ** 2)
                candidates.append({'contour': cnt, 'circularity': circ})
                circularities.append(circ)

    # 2. Calculate MAD bounds for circularity (changed)
    c_low, c_high = get_mad_range(circularities, z_thresh=3.0)

    # 3. Apply MAD filteringchanged)
    final_c_full_list = []
    for cand in candidates:
        if c_low <= cand['circularity'] <= c_high:
            final_c_full_list.append(cand['contour'])

    c_list = list(map(lambda x: min_rect_circle(x, radius), final_c_full_list))
    c_list = [x for x in c_list if x is not None]

    cv_list.append(c_list)
    # print(f"Candidates: {len(candidates)}, After MAD: {len(c_list)}")


In [45]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    #if idx == 6:
    #    break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in cv_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [46]:
!pip install starfile -qq

DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/CRF-0.0.1-py3.12-linux-x86_64.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330


In [47]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in cv_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [48]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1509,3948,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,427,3944,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,2542,3945,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,753,3943,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,3715,3939,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
34014,781,179,00391@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
34015,2953,149,00392@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
34016,316,147,00393@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
34017,2856,158,00394@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [49]:
import starfile
starfile.write(df, f'{RESULT_DIR}/cv_particles.star')

### TrackPy

In [50]:
# cv_list

In [51]:
tp_config

(0.15, 0.4)

In [52]:
import trackpy as tp
from skimage.measure import regionprops, label

In [53]:
TrackParticleSize = RADIUS*2-1
curr_adpmass = 0
sep = round(TrackParticleSize*tp_config[1])
scale = tp_config[0]  # Scale factor (0.5 means reducing the size to half)

In [54]:
tp_list = []
for img in label_images:
    output_image = img
    small_image = cv2.resize(output_image, None, fx=scale, fy=scale, interpolation=cv2.INTER_AREA)
    # Adjust parameters based on the scale
    small_sep = int(sep * scale)
    small_diameter = int(TrackParticleSize * scale)
    if small_diameter % 2 == 0:
        small_diameter += 1

    # 1. Initial Localizations (changed)
    coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)

    if len(coorTrack) > 0:
        # 2. Extract Shape Metrics for MAD (changed)
        # We create a mask from the image to calculate circularity of the objects trackpy found
        mask = (small_image > (small_image.mean())).astype(np.uint8)
        labeled_mask = label(mask)
        props = regionprops(labeled_mask)

        # Map trackpy coordinates to region circularity
        circularities = []
        valid_indices = []

        for idx, row in coorTrack.iterrows():
            # Find the region containing the trackpy coordinate
            py, px = int(row['y']), int(row['x'])
            region_label = labeled_mask[py, px]

            if region_label > 0:
                region = props[region_label - 1]
                area = region.area
                perimeter = region.perimeter
                if perimeter > 0:
                    circ = (4 * np.pi * area) / (perimeter ** 2)
                    circularities.append(circ)
                    valid_indices.append(idx)
                else:
                    circularities.append(0)
                    valid_indices.append(idx)
            else:
                circularities.append(0)
                valid_indices.append(idx)

        # 3. Apply MAD Filtering (changed)
        c_low, c_high = get_mad_range([c for c in circularities if c > 0], z_thresh=3.0)

        coorTrack['circularity'] = circularities
        coorTrack = coorTrack[(coorTrack['circularity'] >= c_low) & (coorTrack['circularity'] <= c_high)]

    """coorTrack = tp.locate(small_image, diameter=small_diameter, minmass=curr_adpmass, separation=small_sep)"""
    #coorTrack.loc[:,'prob']=[output_image[getattr(coor,'y'),getattr(coor,'x')] for coor in np.floor(coorTrack[['x','y']]).astype('int').itertuples()]

    coorTrack['x'] *= (1/scale)
    coorTrack['y'] *= (1/scale)
    coords = coorTrack[['x', 'y']].values
    tp_list.append(coords)
    # print(len(coords))

In [ ]:
# tp_list

In [55]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in tp_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [56]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in tp_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [57]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,467.183199,231.149456,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1139.849686,190.892069,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,3608.337522,191.546208,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,1466.418773,221.543923,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,3140.327007,199.021407,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
40222,765.183429,3831.155733,00464@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40223,1177.581051,3839.509067,00465@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40224,3118.095712,3831.300692,00466@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
40225,1626.322555,3859.998043,00467@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [58]:
import starfile
starfile.write(df, f'{RESULT_DIR}/tp_particles.star')

### Nonmax

In [59]:
import pycuda.driver as drv
from center_finding import cleanCanList, pad_image, reLev, reNorm, scoreGpu, getMax, getMax3, gaussian_kernel_2d_opencv, reshape

In [60]:
nms_config

(0.4, 0.6)

In [61]:
#ratio=1
pnum=2000 #initial filtering, larger if candidate is more
pCan=nms_config[0] #the grid size smaller will generate more candidate
overlap=nms_config[1] #allow for overlap
psize=RADIUS*2

# Nor affect too much
level=3
nSep=10
nIter=100

In [62]:
nms_list = []
for idx, img in enumerate(label_images):
    # if idx == 6:
    #     break
    heatArr=pad_image(img)
    heatArr=reNorm(heatArr, nSep)
    heatArr=reLev(heatArr,level)
    gaus= gaussian_kernel_2d_opencv(kernel_size = psize,sigma = 0)

    # canList,score=scoreGpuLaunch(heatArr,gaus,psize, gsize, nIter, fscore)
    gsize=psize*pCan
    heat=heatArr.astype(np.float32)
    gaus=gaus.astype(np.float32)
    score=np.zeros(heat.shape).astype(np.float32)
    [sizex,sizey]=heat.shape
    sizex = int(sizex)
    sizey = int(sizey)
    psize = int(psize)
    gsize = int(gsize)

    func = scoreGpu

    tx=16
    ty=16
    bx=(sizex-1)//tx+1
    by=(sizey-1)//ty+1
    print('get score gpu',tx, ty, bx, by, gsize, psize)


    func(drv.In(heat), drv.In(gaus), drv.Out(score), np.int32(sizex), np.int32(sizey), np.int32(psize),
        block=(tx, ty, 1), grid=(int(bx), int(by)))

    # # Calculate necessary dimensions and convert all to int
    numx = int((sizex - 1) // gsize + 1)
    numy = int((sizey - 1) // gsize + 1)
    num = numx * numy
    tnum = num * 5

    # # Create the result array
    res = np.zeros(tnum).astype(np.float32)

    # # Block and grid dimensions
    bx = (numx - 1) // tx + 1
    by = (numy - 1) // ty + 1

    # Get the function from the module
    func = getMax

    # Call the function, ensure all parameters are the correct type
    func(drv.In(score), drv.Out(res), np.int32(gsize), np.int32(sizex), np.int32(sizey), np.int32(numx),
        block=(16, 16, 1), grid=(bx, by, 1))

    # Make sure all dimensions and sizes are properly cast to np.int32 to avoid ambiguity
    niter = np.int32(nIter)
    gsize = np.int32(gsize)
    sizex = np.int32(sizex)
    sizey = np.int32(sizey)
    numx = np.int32(numx)
    tnum = np.int32(num)

    print('get Max3', tx, ty, bx, by, niter, 'other : ', gsize, sizex, sizey, numx, tnum)
    func = getMax3
    # Ensuring the correct parameter order and type for the kernel invocation
    func(drv.In(score), drv.InOut(res), gsize, sizex, sizey, numx, tnum, niter,
            block=(16, 16, 1), grid=(bx, by, 1))

    """
    canList=reshape(res,num)

    print('Number of Particles before:', len(canList))
    if(len(canList)>pnum):
        canList=canList[:pnum]


    canList=cleanCanList(canList,overlap,psize)
    #canList=reCan(canList,ratio)
    print('Number of Particles:', len(canList))
    nms_list.append([(r[1],r[0]) for r in canList])
    """
    canList = reshape(res, num)
    if len(canList) > pnum:
        canList = canList[:pnum]

    canList = cleanCanList(canList, overlap, psize)

    # --- Start MAD Circularity Filtering (changed) ---
    if len(canList) > 0:
        # 1. Create a binary mask to analyze shape morphology
        # Using a simple threshold on the processed heatArr
        mask = (heatArr > heatArr.mean()).astype(np.uint8)
        labeled_mask = label(mask)
        props = regionprops(labeled_mask)

        filtered_can_list = []
        circularities = []
        temp_candidates = []

        for r in canList:
            # canList usually returns [y, x, score...] or [x, y, score...]
            # Adjust indexing based on your reshape output (assuming r[0]=y, r[1]=x)
            py, px = int(r[0]), int(r[1])

            # Bounds check for safety
            if 0 <= py < labeled_mask.shape[0] and 0 <= px < labeled_mask.shape[1]:
                region_label = labeled_mask[py, px]
                if region_label > 0:
                    region = props[region_label - 1]
                    area = region.area
                    perim = region.perimeter
                    if perim > 0:
                        circ = (4 * np.pi * area) / (perim ** 2)
                        circularities.append(circ)
                        temp_candidates.append({'data': r, 'circ': circ})

        if circularities:
            # 2. Determine statistical bounds for this specific image
            c_low, c_high = get_mad_range(circularities, z_thresh=3.0)

            # 3. Final selection based on MAD bounds
            for cand in temp_candidates:
                if c_low <= cand['circ'] <= c_high:
                    filtered_can_list.append(cand['data'])

        canList = filtered_can_list
    # --- End MAD Circularity Filtering (changed) ---

    print('Number of Particles after MAD:', len(canList))
    nms_list.append([(r[1], r[0]) for r in canList])

get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 577
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 559
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 564
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 325
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 362
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 553
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 5776
convert
Number of Particles after MAD: 304
get score gpu 16 16 240 240 51 128
get Max3 16 16 5 5 100 other :  51 3840 3840 76 

In [63]:
import matplotlib
for idx, (test_image, _, _, grid, _) in enumerate(full_dataset):
    # if idx == 6:
    #     break
    if idx % 30 == 0:
    # else:
        label_image = label_images[idx]
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)

        for indx in nms_list[idx]:
            x, y = indx
            c = matplotlib.patches.Circle((x, y), radius=RADIUS, fill=False, color='r')
            ax.add_patch(c)
        plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [64]:
import pandas as pd

for idx, fname in enumerate(fnames):
    # Create the adjusted list of tuples for current index
    adjusted_c_list = [(x + BORDER, y + BORDER) for x, y in nms_list[idx]]

    # Create a temporary DataFrame from the list of tuples
    temp_df = pd.DataFrame(adjusted_c_list, columns=['rlnCoordinateX', 'rlnCoordinateY'])
    temp_df['rlnImageName'] = [f"{i:05d}@{fname}.mrc" for i in range(len(temp_df))]
    temp_df['rlnMicrographName'] = fname + ".mrc"

    # If this is the first DataFrame, initialize df
    if idx == 0:
        df = temp_df
    else:
        # Append the temporary DataFrame to the existing df
        df = pd.concat([df, temp_df], ignore_index=True)

In [65]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,1134.6,1778.2,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,2792.6,3101.0,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,2442.8,3780.6,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,2300.4,3764.8,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,353.4,2349.2,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
41297,1524.0,3030.0,00516@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
41298,3697.0,2015.0,00517@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
41299,1603.0,2510.0,00518@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
41300,1506.0,3542.0,00519@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [66]:
starfile.write(df, f'{RESULT_DIR}/nms_particles.star')

---

## watershed

In [67]:
# @title class function define
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from skimage.measure import regionprops
import math

class WatershedDetector:
    def __init__(self, particle_radius=64, border=0, margin=40, z_thresh=3.0, use_contour_area=True):
        self.radius = particle_radius
        self.border = border
        self.margin = margin
        self.z_thresh = z_thresh
        self.use_contour_area = use_contour_area
        self.expected_area = np.pi * (self.radius ** 2)

    def _get_contour_expected_area(self, mask):
        """Calculates expected area based on actual mask contours."""
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours: return self.expected_area
        areas = [cv2.contourArea(cnt) for cnt in contours]
        median_area = np.median(areas)
        return median_area if median_area > 0 else self.expected_area

    def _calculate_metrics(self, region):
        """Generates the full dictionary of shape descriptors."""
        area = region.area
        perimeter = region.perimeter
        if perimeter == 0: return None
        return {
            'area': area,
            'circularity': (4 * np.pi * area) / (perimeter ** 2),
            'eccentricity': region.eccentricity,
            'solidity': area / region.convex_area if region.convex_area > 0 else 0,
            'hu_0': region.moments_hu[0],
            'hu_1': region.moments_hu[1],
            'hu_2': region.moments_hu[2]
        }

    def _get_mad_range(self, data):
        """Calculates the filtering window using Median Absolute Deviation."""
        arr = np.array(data)
        med = np.median(arr)
        mad = np.median(np.abs(arr - med))
        if mad == 0: return np.min(arr), np.max(arr)
        low = med - (self.z_thresh * mad / 0.6745)
        high = med + (self.z_thresh * mad / 0.6745)
        return max(0.0, low), min(1.0, high)

    def _plot_all_histograms(self, candidates, c_bounds):
        """Generates an auto-adjusting grid of histograms for all metrics. (changed)"""
        # Convert candidate metrics list of dicts to a DataFrame for easy plotting
        metrics_df = pd.DataFrame([c['metrics'] for c in candidates])
        metric_names = metrics_df.columns.tolist()

        n_metrics = len(metric_names)
        n_cols = 3
        n_rows = math.ceil(n_metrics / n_cols)

        fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, 4 * n_rows))
        axes = axes.flatten() # Flatten to 1D for easy iteration

        for idx, col_name in enumerate(metric_names):
            ax = axes[idx]
            ax.hist(metrics_df[col_name], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
            ax.set_title(f"Distribution: {col_name}")
            ax.set_xlabel("Value")
            ax.set_ylabel("Frequency")

            # Only add bounds for Circularity as requested (changed)
            if col_name == 'circularity':
                ax.axvline(c_bounds[0], color='red', linestyle='--', linewidth=2, label=f'Lower: {c_bounds[0]:.2f}')
                ax.axvline(c_bounds[1], color='blue', linestyle='--', linewidth=2, label=f'Upper: {c_bounds[1]:.2f}')
                ax.legend(loc='upper right', fontsize='small')

        # Hide any unused axes in the grid
        for i in range(idx + 1, len(axes)):
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()

    def detect(self, prob_map_input, peak_thresh=0.4, dist_ratio=0.8, show_plots=True):
        """Main detection entry point with optional visualization. (changed)"""
        prob_map = prob_map_input.astype(np.float32)
        if prob_map.max() > 1.0: prob_map /= 255.0
        mask = (prob_map > 0.5).astype(np.uint8)
        if np.sum(mask) == 0: mask = (prob_map > 0.05).astype(np.uint8)

        current_expected_area = self._get_contour_expected_area(mask) if self.use_contour_area else self.expected_area

        distance = ndi.distance_transform_edt(mask)
        coords = peak_local_max(distance, min_distance=int(self.radius * dist_ratio),
                                labels=mask, threshold_abs=self.radius * peak_thresh)
        markers = np.zeros(distance.shape, dtype=int)
        for i, (r, c) in enumerate(coords): markers[r, c] = i + 1
        labels = watershed(-distance, markers, mask=mask)

        candidates, circularities = [], []
        for region in regionprops(labels):
            if current_expected_area * 0.3 <= region.area <= current_expected_area * 2.0:
                m = self._calculate_metrics(region)
                if m:
                    candidates.append({'region': region, 'metrics': m})
                    circularities.append(m['circularity'])

        if not candidates: return []

        c_bounds = self._get_mad_range(circularities)

        # Trigger histogram plotting if enabled (changed)
        if show_plots:
            self._plot_all_histograms(candidates, c_bounds)

        final_particles = []
        for cand in candidates:
            if c_bounds[0] <= cand['metrics']['circularity'] <= c_bounds[1]:
                ry, rx = cand['region'].centroid
                if (self.margin < rx < prob_map.shape[1] - self.margin and
                    self.margin < ry < prob_map.shape[0] - self.margin):

                    final_particles.append({
                        'x': int(rx) + self.border,
                        'y': int(ry) + self.border,
                        'score': float(prob_map[int(ry), int(rx)]),
                        **cand['metrics']
                    })
        return final_particles

In [68]:
# @title execute watershed postprocessing

 # Control histograms
SHOW_PLOTS_A = True #@param {type:"boolean"}
# Control overlays
SHOW_PLOTS_B = True #@param {type:"boolean"}
# show plot per N micrograph to show plot
per_N_show = 30 #@param {type:"integer"}

detector = WatershedDetector(particle_radius=RADIUS, border=BORDER)
all_star_rows, all_metrics_rows = [], []

for idx, (test_image, _, _, grid, _) in enumerate(tqdm(full_dataset, desc="Processing Images")):
    fname = fnames[idx]
    label_image = label_images[idx]

    # 1. Detect particles and plot histograms (inside class)
    particles = detector.detect(label_image, peak_thresh=ws_config[0],
                                dist_ratio=ws_config[1], show_plots=SHOW_PLOTS_A and (idx % per_N_show == 0))

    if not particles: continue

    # 2. Plot Micrograph Overlay Plot
    if SHOW_PLOTS_B and idx % per_N_show == 0:
        _, ax = plt.subplots(figsize=(12, 12))
        ax.imshow(processed_micrographs[idx], cmap='gray')
        ax.imshow(label_image, cmap='inferno', alpha=0.4)
        for p in particles:
            # Note: Subtracting BORDER for plot alignment if needed
            circ = plt.Circle((p['x']-BORDER, p['y']-BORDER), radius=RADIUS, fill=False, color='r')
            ax.add_patch(circ)
        ax.set_title(f"Overlay Verification: {fname} (idx: {idx})")
        plt.show()

    # 3. Data Collection
    for i, p in enumerate(particles):
        p['rlnMicrographName'] = fname + ".mrc"
        p['particle_id'] = f"{i:05d}@{fname}.mrc"
        # Explicitly add RELION coordinates to metrics DF
        p['rlnCoordinateX'] = p.pop('x')
        p['rlnCoordinateY'] = p.pop('y')
        all_metrics_rows.append(p)

        all_star_rows.append({
            'rlnCoordinateX': p['rlnCoordinateX'], 'rlnCoordinateY': p['rlnCoordinateY'],
            'rlnImageName': p['particle_id'], 'rlnMicrographName': p['rlnMicrographName']
        })

# 4. Save Final CSV
df = pd.DataFrame(all_star_rows) # For STAR file
df_metrics = pd.DataFrame(all_metrics_rows) # For Shape CSV (changed)

# Reorder columns for visibility
coord_cols = ['particle_id', 'rlnCoordinateX', 'rlnCoordinateY', 'score', 'area', 'circularity']
df_metrics = df_metrics[coord_cols + [c for c in df_metrics.columns if c not in coord_cols]]

df_metrics_sorted = df_metrics.sort_values(
    by=['rlnMicrographName', 'rlnCoordinateX', 'rlnCoordinateY'],
    ascending=True
)

parent_of_result_dir = os.path.dirname(RESULT_DIR)
csv_path = os.path.join(parent_of_result_dir, "particles_dfcsv_rst", "particle_shape_analysis.csv")
os.makedirs(os.path.join(parent_of_result_dir, "particles_dfcsv_rst"), exist_ok=True)
df_metrics_sorted.to_csv(csv_path, index=False)

print(f"Final Data Generation Complete: {len(df)} particles.")

Output hidden; open in https://colab.research.google.com to view.

In [69]:
df

,rlnCoordinateX,rlnCoordinateY,rlnImageName,rlnMicrographName
0,549,1206,00000@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
1,1771,418,00001@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
2,2265,843,00002@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
3,217,2979,00003@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
4,2078,3253,00004@Falcon_2012_06_12-15_43_48_0.mrc,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...
42900,2651,836,00451@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
42901,2388,3372,00452@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
42902,3926,2925,00453@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc
42903,1803,1313,00454@Falcon_2012_06_13-10_27_05_0.mrc,Falcon_2012_06_13-10_27_05_0.mrc


In [70]:
df_metrics

,particle_id,rlnCoordinateX,rlnCoordinateY,score,area,circularity,eccentricity,solidity,hu_0,hu_1,hu_2,rlnMicrographName
0,00000@Falcon_2012_06_12-15_43_48_0.mrc,549,1206,0.996078,6694.0,0.811268,0.419787,0.948159,0.163021,0.000248,0.000118,Falcon_2012_06_12-15_43_48_0.mrc
1,00001@Falcon_2012_06_12-15_43_48_0.mrc,1771,418,0.996078,6634.0,0.852765,0.330868,0.971161,0.159920,0.000086,0.000003,Falcon_2012_06_12-15_43_48_0.mrc
2,00002@Falcon_2012_06_12-15_43_48_0.mrc,2265,843,0.996078,7446.0,0.837333,0.513859,0.969027,0.162100,0.000608,0.000016,Falcon_2012_06_12-15_43_48_0.mrc
3,00003@Falcon_2012_06_12-15_43_48_0.mrc,217,2979,0.996078,7077.0,0.854132,0.420130,0.977216,0.161718,0.000245,0.000117,Falcon_2012_06_12-15_43_48_0.mrc
4,00004@Falcon_2012_06_12-15_43_48_0.mrc,2078,3253,0.996078,6441.0,0.886239,0.472585,0.983960,0.161317,0.000411,0.000031,Falcon_2012_06_12-15_43_48_0.mrc
...,...,...,...,...,...,...,...,...,...,...,...,...
42900,00451@Falcon_2012_06_13-10_27_05_0.mrc,2651,836,0.996078,2810.0,0.762219,0.669245,0.940114,0.174982,0.002550,0.000695,Falcon_2012_06_13-10_27_05_0.mrc
42901,00452@Falcon_2012_06_13-10_27_05_0.mrc,2388,3372,0.996078,2820.0,0.679254,0.760068,0.935944,0.180084,0.005350,0.000315,Falcon_2012_06_13-10_27_05_0.mrc
42902,00453@Falcon_2012_06_13-10_27_05_0.mrc,3926,2925,0.996078,3986.0,0.808186,0.774781,0.980807,0.179370,0.005917,0.000126,Falcon_2012_06_13-10_27_05_0.mrc
42903,00454@Falcon_2012_06_13-10_27_05_0.mrc,1803,1313,0.996078,3056.0,0.707408,0.868678,0.955896,0.203926,0.015267,0.000352,Falcon_2012_06_13-10_27_05_0.mrc


In [71]:
df_metrics_sorted

,particle_id,rlnCoordinateX,rlnCoordinateY,score,area,circularity,eccentricity,solidity,hu_0,hu_1,hu_2,rlnMicrographName
4058,00172@Falcon_2012_06_12-14_33_35_0.mrc,178,1672,0.996078,5322.0,0.854984,0.480603,0.978669,0.163659,0.000457,0.000072,Falcon_2012_06_12-14_33_35_0.mrc
4160,00274@Falcon_2012_06_12-14_33_35_0.mrc,197,355,0.996078,5914.0,0.750440,0.770352,0.949888,0.177210,0.005590,0.000050,Falcon_2012_06_12-14_33_35_0.mrc
4399,00513@Falcon_2012_06_12-14_33_35_0.mrc,210,2716,0.996078,5179.0,0.782735,0.808384,0.973496,0.182802,0.007871,0.000003,Falcon_2012_06_12-14_33_35_0.mrc
4176,00290@Falcon_2012_06_12-14_33_35_0.mrc,219,3872,0.996078,5220.0,0.847861,0.669090,0.978628,0.168641,0.002365,0.000205,Falcon_2012_06_12-14_33_35_0.mrc
4328,00442@Falcon_2012_06_12-14_33_35_0.mrc,220,1374,0.996078,5961.0,0.764432,0.807387,0.963783,0.184097,0.007924,0.000068,Falcon_2012_06_12-14_33_35_0.mrc
...,...,...,...,...,...,...,...,...,...,...,...,...
42887,00438@Falcon_2012_06_13-10_27_05_0.mrc,3922,357,0.996078,3083.0,0.725069,0.541386,0.922501,0.173541,0.000888,0.000404,Falcon_2012_06_13-10_27_05_0.mrc
42892,00443@Falcon_2012_06_13-10_27_05_0.mrc,3924,3137,0.996078,3349.0,0.794690,0.604787,0.974680,0.172885,0.001497,0.000919,Falcon_2012_06_13-10_27_05_0.mrc
42895,00446@Falcon_2012_06_13-10_27_05_0.mrc,3926,2853,0.996078,3368.0,0.859680,0.445771,0.976798,0.162561,0.000322,0.000089,Falcon_2012_06_13-10_27_05_0.mrc
42902,00453@Falcon_2012_06_13-10_27_05_0.mrc,3926,2925,0.996078,3986.0,0.808186,0.774781,0.980807,0.179370,0.005917,0.000126,Falcon_2012_06_13-10_27_05_0.mrc


In [72]:
# @title store star file watershed
watershed_star_name = 'watershed_particles.star' # @param{type: "string"}
starfile.write(df, f'{RESULT_DIR}/{watershed_star_name}')